# 🌱 Colab Roots v2
### The most stable ephemeral VM on Google Colab

This notebook bootstraps a persistent development environment with:
- 🖥️ **code-server** — Full VS Code IDE in your browser
- 💻 **ttyd** — Browser-based terminal (tmux-backed)
- 🔧 **VS Code Remote Tunnel** — Connect from desktop VS Code
- 🔄 **Daemon** — Process management, health checks, keep-alive
- ☁️ **Google Drive persistence** — Auto-sync your work

**⚠️ Disclaimer**: For educational and development purposes. Use responsibly within Google Colab's Terms of Service.

## ⚙️ Configuration

In [ ]:
#@title 🔧 Setup Configuration{display-mode: "form"}
#@markdown ### Workspace
WORKSPACE_REPO = ""  #@param {type:"string"}
WORKSPACE_BRANCH = "main"  #@param {type:"string"}

#@markdown ### Services
ENABLE_VSCODE_TUNNEL = True  #@param {type:"boolean"}
ENABLE_CODE_SERVER = True  #@param {type:"boolean"}
ENABLE_TTYD = True  #@param {type:"boolean"}

#@markdown ### Persistence
PERSIST_TO_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "colab-roots"  #@param {type:"string"}

#@markdown ### Keep-Alive
KEEP_ALIVE = True  #@param {type:"boolean"}
KEEP_ALIVE_INTERVAL = 300  #@param {type:"integer"}

#@markdown ---
import os, secrets, string

# Generate cryptographically secure password
CHARS = string.ascii_letters + string.digits
PASSWORD = ''.join(secrets.choice(CHARS) for _ in range(20))
USERNAME = "roots"
TUNNEL_NAME = f"colab-roots-{secrets.token_hex(4)}"

# Set environment variables for later cells
os.environ["ROOTS_PASSWORD"] = PASSWORD
os.environ["ROOTS_USERNAME"] = USERNAME
os.environ["ROOTS_TUNNEL"] = TUNNEL_NAME
os.environ["ROOTS_DRIVE_FOLDER"] = DRIVE_FOLDER

# Derived paths
ROOTS_HOME = os.path.expanduser("~/.colab-roots")
ROOTS_BIN = os.path.join(ROOTS_HOME, "bin")
ROOTS_LOGS = os.path.join(ROOTS_HOME, "logs")
ROOTS_STATE = os.path.join(ROOTS_HOME, "state")

os.environ["ROOTS_HOME"] = ROOTS_HOME
os.environ["ROOTS_BIN"] = ROOTS_BIN
os.environ["ROOTS_LOGS"] = ROOTS_LOGS
os.environ["ROOTS_STATE"] = ROOTS_STATE
os.environ["ROOTS_WORKSPACE"] = "/content/workspace"

print("✅ Configuration loaded")
print(f"   Username:     {USERNAME}")
print(f"   Password:     {PASSWORD}")
print(f"   Tunnel name:  {TUNNEL_NAME}")
print(f"   Roots home:   {ROOTS_HOME}")

## 🚀 Bootstrap
Run the cell below to install everything and start all services.

In [ ]:
#@title 🌱 Start Colab Roots{display-mode: "form"}
import os, subprocess, sys, time, json, secrets
from pathlib import Path

# Re-read config from environment
ROOTS_HOME = Path(os.environ.get("ROOTS_HOME", "~/.colab-roots")).expanduser()
ROOTS_BIN = ROOTS_HOME / "bin"
ROOTS_LOGS = ROOTS_HOME / "logs"
ROOTS_STATE = ROOTS_HOME / "state"
PASSWORD = os.environ.get("ROOTS_PASSWORD", secrets.token_urlsafe(16))
USERNAME = os.environ.get("ROOTS_USERNAME", "roots")
TUNNEL_NAME = os.environ.get("ROOTS_TUNNEL", "colab-roots")
WORKSPACE = Path("/content/workspace")

print("═══════════════════════════════════════════")
print("  🌱 Colab Roots — Bootstrap v2.0")
print("═══════════════════════════════════════════")
print()

# ─── Create directories ─────────────────────────────────────────
for d in (ROOTS_HOME, ROOTS_BIN, ROOTS_LOGS, ROOTS_STATE):
    d.mkdir(parents=True, exist_ok=True)

# ─── Helper: run shell command ──────────────────────────────────
def sh(cmd, check=False, capture=True):
    """Run a shell command safely."""
    result = subprocess.run(
        cmd, shell=isinstance(cmd, str),
        capture_output=capture, text=True
    )
    if check and result.returncode != 0:
        print(f"   ⚠️  Command failed: {cmd}")
        if result.stderr:
            print(f"      {result.stderr[:200]}")
    return result

# ─── Step 1: System dependencies ──────────────────────────────
print("📦 [1/6] Installing system dependencies...")
sh("apt-get update -qq")
sh("apt-get install -y -qq curl wget git tmux jq rsync")
print("   ✅ System packages installed")

# ─── Step 2: code-server ──────────────────────────────────────
if ENABLE_CODE_SERVER:
    print("📦 [2/6] Installing code-server...")
    if not sh("command -v code-server").returncode == 0:
        sh("curl -fsSL https://code-server.dev/install.sh | sh")
    print("   ✅ code-server installed")
else:
    print("⏭️  [2/6] code-server skipped")

# ─── Step 3: ttyd ─────────────────────────────────────────────
if ENABLE_TTYD:
    print("📦 [3/6] Installing ttyd...")
    if not sh("command -v ttyd").returncode == 0:
        # Try multiple sources
        ok = sh("wget -q https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64 -O /usr/local/bin/ttyd").returncode == 0
        if not ok:
            sh("apt-get install -y -qq ttyd")
        sh("chmod +x /usr/local/bin/ttyd", check=False)
    print("   ✅ ttyd installed")
else:
    print("⏭️  [3/6] ttyd skipped")

# ─── Step 4: cloudflared ──────────────────────────────────────
print("📦 [4/6] Installing cloudflared...")
if sh("command -v cloudflared").returncode != 0:
    sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
    sh("chmod +x /usr/local/bin/cloudflared", check=False)
print("   ✅ cloudflared installed")

# ─── Step 5: rclone ───────────────────────────────────────────
if PERSIST_TO_DRIVE:
    print("📦 [5/6] Installing rclone...")
    if sh("command -v rclone").returncode != 0:
        sh("curl -fsSL https://rclone.org/install.sh | bash")
    print("   ✅ rclone installed")
else:
    print("⏭️  [5/6] rclone skipped")

# ─── Step 6: VS Code tunnel ──────────────────────────────────
if ENABLE_VSCODE_TUNNEL:
    print("📦 [6/6] Installing VS Code tunnel...")
    if sh("command -v code-tunnel").returncode != 0:
        # Download VS Code CLI with tunnel support
        sh("wget -q 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' -O /tmp/vscode-cli.tar.gz")
        sh("mkdir -p /tmp/vscode-cli-extract")
        sh("tar -xzf /tmp/vscode-cli.tar.gz -C /tmp/vscode-cli-extract")
        sh("cp /tmp/vscode-cli-extract/code /usr/local/bin/code-tunnel && chmod +x /usr/local/bin/code-tunnel", check=False)
        sh("rm -rf /tmp/vscode-cli*", check=False)
    if sh("command -v code-tunnel").returncode == 0:
        print("   ✅ VS Code tunnel installed")
    else:
        print("   ⚠️  VS Code tunnel install failed (optional)")
else:
    print("⏭️  [6/6] VS Code tunnel skipped")

# ─── Google Drive ─────────────────────────────────────────────
DRIVE_PATH = ""
if PERSIST_TO_DRIVE:
    print()
    print("☁️  Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_PATH = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    os.makedirs(DRIVE_PATH, exist_ok=True)
    os.makedirs(f"{DRIVE_PATH}/workspace", exist_ok=True)
    print(f"   ✅ Drive mounted at {DRIVE_PATH}")

    # Restore state from previous session
    state_dir = Path(DRIVE_PATH) / "state"
    if state_dir.exists():
        print("   📥 Restoring state from Drive...")
        for f in state_dir.iterdir():
            if f.is_file():
                dest = ROOTS_STATE / f.name
                dest.write_bytes(f.read_bytes())
        print("   ✅ State restored")

    # Restore packages
    req_file = Path(DRIVE_PATH) / "requirements.txt"
    if req_file.exists():
        print("   📦 Restoring packages...")
        sh(f"pip install -q -r '{req_file}'")
        print("   ✅ Packages restored")

# ─── Clone workspace repo ─────────────────────────────────────
if WORKSPACE_REPO:
    print()
    print(f"📥 Cloning workspace: {WORKSPACE_REPO}")
    if (WORKSPACE / ".git").exists():
        sh(f"cd {WORKSPACE} && git pull --ff-only", check=False)
    else:
        import shutil
        if WORKSPACE.exists():
            shutil.rmtree(WORKSPACE)
        result = sh(f"git clone --branch {WORKSPACE_BRANCH} {WORKSPACE_REPO} {WORKSPACE}")
        if result.returncode != 0:
            # Fallback: try without branch
            sh(f"git clone {WORKSPACE_REPO} {WORKSPACE}", check=False)
    print(f"   ✅ Workspace ready at {WORKSPACE}")

# ─── Write password file for code-server ──────────────────────
pw_file = ROOTS_STATE / "code-server-pw"
pw_file.write_text(PASSWORD)
pw_file.chmod(0o600)
(ROOTS_STATE / "password").write_text(PASSWORD)
(ROOTS_STATE / "password").chmod(0o600)
(ROOTS_STATE / "tunnel_name").write_text(TUNNEL_NAME)

# ─── Start tmux session ──────────────────────────────────────
print()
print("🔧 Starting services...")
sh("tmux kill-session -t roots 2>/dev/null || true")
sh("tmux new-session -d -s roots -x 220 -y 50")
print("   ✅ tmux session 'roots' started")

# ─── Start code-server ───────────────────────────────────────
if ENABLE_CODE_SERVER:
    sh(f"nohup code-server --bind-addr 127.0.0.1:8080 --auth password --password-file '{pw_file}' --disable-telemetry --disable-update-check /content/workspace > '{ROOTS_LOGS}/code-server.log' 2>&1 &")
    print("   ✅ code-server running on 127.0.0.1:8080")

# ─── Start ttyd ──────────────────────────────────────────────
if ENABLE_TTYD:
    sh(f"nohup ttyd -p 7681 -W -c '{USERNAME}:{PASSWORD}' tmux attach -t roots > '{ROOTS_LOGS}/ttyd.log' 2>&1 &")
    print("   ✅ ttyd running on 127.0.0.1:7681")

# ─── Start daemon ────────────────────────────────────────────
print()
print("🤖 Starting Colab Roots daemon...")

# Copy daemon to persistent location
daemon_src = Path("/content/.colab-roots/src/daemon.py")
daemon_dst = ROOTS_HOME / "src" / "daemon.py"
daemon_dst.parent.mkdir(parents=True, exist_ok=True)
if daemon_src.exists():
    daemon_dst.write_text(daemon_src.read_text())

# Start daemon as background process
daemon_env = os.environ.copy()
daemon_env.update({
    "ROOTS_HOME": str(ROOTS_HOME),
    "ROOTS_LOGS": str(ROOTS_LOGS),
    "ROOTS_STATE": str(ROOTS_STATE),
    "ROOTS_PASSWORD": PASSWORD,
    "ROOTS_USERNAME": USERNAME,
    "ROOTS_TUNNEL": TUNNEL_NAME,
    "ROOTS_DRIVE_PATH": DRIVE_PATH,
    "ROOTS_ENABLE_CODE_SERVER": str(ENABLE_CODE_SERVER),
    "ROOTS_ENABLE_TTYD": str(ENABLE_TTYD),
    "ROOTS_ENABLE_VSCODE_TUNNEL": str(ENABLE_VSCODE_TUNNEL),
    "ROOTS_PERSIST_TO_DRIVE": str(PERSIST_TO_DRIVE),
    "ROOTS_KEEP_ALIVE": str(KEEP_ALIVE),
    "ROOTS_KEEP_ALIVE_INTERVAL": str(KEEP_ALIVE_INTERVAL),
})

# Launch daemon via Python
daemon_script = f"""
import os, sys
sys.path.insert(0, '{daemon_dst.parent}')
from daemon import ColabRootsDaemon

config = {{
    'roots_home': os.environ['ROOTS_HOME'],
    'enable_code_server': os.environ.get('ROOTS_ENABLE_CODE_SERVER', 'True') == 'True',
    'enable_ttyd': os.environ.get('ROOTS_ENABLE_TTYD', 'True') == 'True',
    'enable_vscode_tunnel': os.environ.get('ROOTS_ENABLE_VSCODE_TUNNEL', 'False') == 'True',
    'persist_to_drive': os.environ.get('ROOTS_PERSIST_TO_DRIVE', 'False') == 'True',
    'drive_path': os.environ.get('ROOTS_DRIVE_PATH', ''),
    'keep_alive': os.environ.get('ROOTS_KEEP_ALIVE', 'True') == 'True',
    'keep_alive_interval': int(os.environ.get('ROOTS_KEEP_ALIVE_INTERVAL', '300')),
    'username': os.environ.get('ROOTS_USERNAME', 'roots'),
    'password': os.environ.get('ROOTS_PASSWORD', ''),
    'tunnel_name': os.environ.get('ROOTS_TUNNEL', 'colab-roots'),
}}

daemon = ColabRootsDaemon(config)
daemon.start_background()
print('   ✅ Daemon started')

# Keep daemon alive in background
import time
while daemon._running:
    time.sleep(60)
"""

sh(f"nohup python3 -c '{daemon_script}' > '{ROOTS_LOGS}/daemon.log' 2>&1 &", check=False)
time.sleep(2)  # Let daemon start

# ─── Print results ──────────────────────────────────────────
print()
print("═══════════════════════════════════════════")
print("  🌱 Colab Roots is LIVE")
print("═══════════════════════════════════════════")
print()
if ENABLE_CODE_SERVER:
    print(f"🖥️  IDE (code-server): http://127.0.0.1:8080")
    print(f"   Password: {PASSWORD}")
    print()
if ENABLE_TTYD:
    print(f"💻 Terminal (ttyd): http://127.0.0.1:7681")
    print(f"   User: {USERNAME}  Password: {PASSWORD}")
    print()
if ENABLE_VSCODE_TUNNEL:
    print(f"🔧 VS Code Remote Tunnel: {TUNNEL_NAME}")
    print(f"   (Use VS Code Remote Tunnels extension to connect)")
    print()
print(f"📋 Manage:  export PATH=\"{ROOTS_BIN}:\$PATH\" && roots status")
print()

## 🔗 Expose via Cloudflare Tunnel

Creates a **temporary** public URL for your services. No Cloudflare account needed.

In [ ]:
#@title 🌐 Create Cloudflare Tunnel{display-mode: "form"}
#@markdown Which service to expose:
EXPOSE_SERVICE = "code-server"  #@param ["code-server", "ttyd", "both"]

import subprocess, re, threading, os

def start_tunnel(port):
    """Start a quick cloudflare tunnel and capture the URL."""
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    url = None
    for line in proc.stdout:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if match:
            url = match.group(1)
            break
    return proc, url

# Read password from state file (not env var — avoids leaking)
pw_file = os.path.expanduser("~/.colab-roots/state/password")
password = open(pw_file).read().strip() if os.path.exists(pw_file) else "UNKNOWN"

if EXPOSE_SERVICE in ('code-server', 'both'):
    print("🌐 Starting tunnel for code-server (port 8080)...")
    proc, url = start_tunnel(8080)
    if url:
        print(f"   ✅ code-server: {url}")
        print(f"   Password: {password}")
    else:
        print("   ⚠️  Could not capture URL")

if EXPOSE_SERVICE in ('ttyd', 'both'):
    print("🌐 Starting tunnel for ttyd (port 7681)...")
    proc, url = start_tunnel(7681)
    if url:
        print(f"   ✅ ttyd: {url}")
        print(f"   User/Password: roots / {password}")
    else:
        print("   ⚠️  Could not capture URL")

print()
print("⚠️  These tunnels are temporary and stop when this cell is interrupted.")
print("   For persistent tunnels, use VS Code Remote Tunnels instead.")

## 💓 Keep-Alive
Prevents idle disconnection by running a background heartbeat.

In [ ]:
#@title 💓 Start Keep-Alive{display-mode: "form"}
#@markdown Interval in seconds (minimum 60):
INTERVAL = 300  #@param {type:"integer"}

import time, threading, datetime, signal as _signal

_keepalive_stop = threading.Event()

def _keepalive_loop():
    """Periodically execute lightweight operations to prevent idle timeout."""
    iteration = 0
    while not _keepalive_stop.is_set():
        iteration += 1
        now = datetime.datetime.now().strftime('%H:%M:%S')
        _ = sum(range(1000))  # CPU activity
        _ = datetime.datetime.now().isoformat()  # I/O activity
        # Write heartbeat file
        try:
            hb = os.path.expanduser("~/.colab-roots/state/heartbeat")
            with open(hb, 'w') as f:
                f.write(datetime.datetime.now().isoformat())
        except Exception:
            pass
        print(f"[{now}] 💓 Beat #{iteration}")
        _keepalive_stop.wait(max(int(INTERVAL), 60))

t = threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive")
t.start()
print(f"✅ Keep-alive started (interval: {max(int(INTERVAL), 60)}s)")
print("   Runs in background. Interrupt this cell to stop.")

## 🔄 Status & Reconnect

In [ ]:
#@title 🔍 Check Status{display-mode: "form"}
import os, subprocess

ROOTS_HOME = os.path.expanduser("~/.colab-roots")
ROOTS_STATE = os.path.join(ROOTS_HOME, "state")

print("═══════════════════════════════════════════")
print("  🌱 Colab Roots — Status")
print("═══════════════════════════════════════════")
print()

# Check tmux
result = subprocess.run(["tmux", "has-session", "-t", "roots"], capture_output=True)
print(f"{'✅' if result.returncode == 0 else '❌'} tmux session 'roots': {'ACTIVE' if result.returncode == 0 else 'NOT FOUND'}")

# Check code-server
result = subprocess.run(["pgrep", "-f", "code-server"], capture_output=True)
running_cs = result.returncode == 0
print(f"{'✅' if running_cs else '❌'} code-server: {'RUNNING' if running_cs else 'NOT RUNNING'}")
if running_cs:
    print("   URL: http://127.0.0.1:8080")

# Check ttyd
result = subprocess.run(["pgrep", "-f", "ttyd"], capture_output=True)
running_ttyd = result.returncode == 0
print(f"{'✅' if running_ttyd else '❌'} ttyd: {'RUNNING' if running_ttyd else 'NOT RUNNING'}")
if running_ttyd:
    print("   URL: http://127.0.0.1:7681")

# Check daemon
result = subprocess.run(["pgrep", "-f", "colab_roots_daemon\|daemon.start_background"], capture_output=True)
running_daemon = result.returncode == 0
print(f"{'✅' if running_daemon else '❌'} daemon: {'RUNNING' if running_daemon else 'NOT RUNNING'}")

# Credentials
pw_file = os.path.join(ROOTS_STATE, "password")
if os.path.exists(pw_file):
    print()
    pw = open(pw_file).read().strip()
    print(f"🔑 Password: {pw}")

# System info
print()
print("📊 System:")
result = subprocess.run(["nproc"], capture_output=True, text=True)
print(f"   CPU: {result.stdout.strip()} cores")
result = subprocess.run(["free", "-h"], capture_output=True, text=True)
for line in result.stdout.split("\n"):
    if line.startswith("Mem:"):
        parts = line.split()
        print(f"   RAM: {parts[1]} total, {parts[2]} used")
        break
result = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
for line in result.stdout.split("\n"):
    if line.startswith("/"):
        parts = line.split()
        print(f"   Disk: {parts[3]} free")
        break

## ☁️ Manual Drive Sync

In [ ]:
#@title ☁️ Sync to Google Drive{display-mode: "form"}
import os, subprocess, glob

DRIVE_FOLDER = os.environ.get("ROOTS_DRIVE_FOLDER", "colab-roots")
DRIVE_PATH = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
WORKSPACE = "/content/workspace"
ROOTS_STATE = os.path.expanduser("~/.colab-roots/state")

print("☁️  Syncing to Google Drive...")

if not os.path.isdir("/content/drive/MyDrive"):
    print("❌ Google Drive not mounted. Run the Bootstrap cell first.")
else:
    os.makedirs(f"{DRIVE_PATH}/workspace", exist_ok=True)

    if os.path.isdir(WORKSPACE):
        result = subprocess.run(
            ["rsync", "-av", "--delete",
             "--exclude=.git", "--exclude=node_modules", "--exclude=__pycache__",
             "--exclude=*.log", "--exclude=.env",
             f"{WORKSPACE}/", f"{DRIVE_PATH}/workspace/"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print("✅ Workspace synced")
        else:
            print(f"⚠️  rsync warnings: {result.stderr[:200]}")
    else:
        print("⚠️  No workspace directory found")

    # Sync state (excluding secrets)
    state_dir = f"{DRIVE_PATH}/state"
    os.makedirs(state_dir, exist_ok=True)
    sensitive = {"password", "code-server-pw"}
    for f in glob.glob(f"{ROOTS_STATE}/*"):
        basename = os.path.basename(f)
        if basename not in sensitive and os.path.isfile(f):
            subprocess.run(["cp", f, f"{state_dir}/{basename}"], capture_output=True)
    print("✅ State files synced")

    # Save package list
    result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
    if result.returncode == 0:
        with open(f"{DRIVE_PATH}/requirements.txt", "w") as f:
            f.write(result.stdout)
    print("✅ Package list saved")

    print()
    print("📁 Drive contents:")
    for item in sorted(os.listdir(DRIVE_PATH)):
        path = os.path.join(DRIVE_PATH, item)
        size = os.path.getsize(path) if os.path.isfile(path) else 0
        print(f"   {item} ({size:,} bytes)" if os.path.isfile(path) else f"   {item}/")

## 🛑 Shutdown

In [ ]:
#@title 🛑 Shutdown Colab Roots{display-mode: "form"}
import subprocess, os, signal

print("🛑 Shutting down Colab Roots...")

# Stop keep-alive if running
try:
    _keepalive_stop.set()
    print("   ✅ keep-alive stopped")
except NameError:
    pass

# Send SIGTERM to code-server processes
result = subprocess.run(["pgrep", "-f", "code-server"], capture_output=True, text=True)
if result.returncode == 0:
    for pid in result.stdout.strip().split():
        try:
            os.kill(int(pid), signal.SIGTERM)
        except (ProcessLookupError, ValueError):
            pass
    print("   ✅ code-server stopped")
else:
    print("   ⏭️  code-server not running")

# Send SIGTERM to ttyd processes
result = subprocess.run(["pgrep", "-f", "ttyd"], capture_output=True, text=True)
if result.returncode == 0:
    for pid in result.stdout.strip().split():
        try:
            os.kill(int(pid), signal.SIGTERM)
        except (ProcessLookupError, ValueError):
            pass
    print("   ✅ ttyd stopped")
else:
    print("   ⏭️  ttyd not running")

# Send SIGTERM to cloudflared
result = subprocess.run(["pgrep", "-f", "cloudflared"], capture_output=True, text=True)
if result.returncode == 0:
    for pid in result.stdout.strip().split():
        try:
            os.kill(int(pid), signal.SIGTERM)
        except (ProcessLookupError, ValueError):
            pass
    print("   ✅ cloudflared stopped")
else:
    print("   ⏭️  cloudflared not running")

# Kill tmux session
result = subprocess.run(["tmux", "kill-session", "-t", "roots"], capture_output=True)
print(f"   {'✅ tmux session killed' if result.returncode == 0 else '⏭️  tmux session not found'}")

print()
print("🌱 Colab Roots shut down. Goodbye!")

---

## 📖 CLI Reference

```bash
export PATH="$HOME/.colab-roots/bin:$PATH"
roots status        # Show all service statuses
roots doctor        # Health check
roots restart ttyd  # Restart a service
roots logs daemon   # View daemon logs
roots urls          # Show access URLs
roots password      # Show password (masked)
roots sync          # Force Drive sync
roots down          # Graceful shutdown
```